In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import genpareto

In [ ]:
### Parameters for
u = 0.025186
xi = 0.243224
beta = 0.009486

threshold_quantile = 0.975
n_exceedances = 164

print("Verified EVT Model\n")

print(f"Threshold quantile: {threshold_quantile:.2%}")
print(f"Threshold (u):      {u:.4%}")
print(f"Exceedances:        {n_exceedances}")
print(f"Shape (xi):         {xi:.6f}")
print(f"Scale (beta):       {beta:.6f}")

Verified EVT Model

Threshold quantile: 97.50%
Threshold (u):      2.5186%
Exceedances:        164
Shape (xi):         0.243224
Scale (beta):       0.009486


In [ ]:
n_sim = 100_000

np.random.seed(42)

# Simulate GPD excesses
simulated_excesses = genpareto.rvs(
    c=xi,
    loc=0,
    scale=beta,
    size=n_sim
)

# Convert excesses into total losses
simulated_losses = u + simulated_excesses

df=pd.DataFrame(simulated_losses)

print("EVT Monte Carlo Simulation\n")
print(f"Number of simulations: {n_sim:,}")
print(f"Minimum simulated loss: {simulated_losses.min():.4%}")
print(f"Maximum simulated loss: {simulated_losses.max():.4%}")
print(f"Mean simulated loss:    {simulated_losses.mean():.4%}")

          0
0  0.029901
1  0.067288
2  0.039909
3  0.034883
4  0.026829
EVT Monte Carlo Simulation

Number of simulations: 100,000
Minimum simulated loss: 2.5186%
Maximum simulated loss: 66.4338%
Mean simulated loss:    3.7640%


In [ ]:
simulation_quantiles = [0.90, 0.95, 0.99, 0.995, 0.999, 0.9999]

simulation_results = []

for q in simulation_quantiles:
    simulation_results.append({
        "Quantile": q,
        "Simulated Loss": np.quantile(simulated_losses, q)
    })

simulation_results = pd.DataFrame(simulation_results)

simulation_results["Simulated Loss %"] = (
    simulation_results["Simulated Loss"] * 100
)

print(simulation_results)

   Quantile  Simulated Loss  Simulated Loss %
0    0.9000        0.054174          5.417390
1    0.9500        0.066526          6.652607
2    0.9900        0.105022         10.502245
3    0.9950        0.126035         12.603537
4    0.9990        0.188260         18.825980
5    0.9999        0.328171         32.817122


In [ ]:

# CRASH PROBABILITY ANALYSIS


loss_thresholds = [0.03, 0.05, 0.075, 0.10, 0.15, 0.20]

probability_results = []

# Probability of entering the EVT tail
tail_probability = 1 - threshold_quantile

for threshold in loss_thresholds:

    # Conditional probability:
    # P(L > threshold | L > u)
    conditional_probability = np.mean(
        simulated_losses > threshold
    )

    # Unconditional probability:
    # P(L > threshold)
    unconditional_probability = (
        tail_probability * conditional_probability
    )

    probability_results.append({
        "Loss Threshold": threshold,
        "Conditional Probability": conditional_probability,
        "Unconditional Probability": unconditional_probability
    })

probability_results = pd.DataFrame(probability_results)

probability_results["Conditional Probability %"] = (
    probability_results["Conditional Probability"] * 100
)

probability_results["Unconditional Probability %"] = (
    probability_results["Unconditional Probability"] * 100
)

print(probability_results)

   Loss Threshold  Conditional Probability  Unconditional Probability  \
0           0.030                  0.61928                   0.015482   
1           0.050                  0.13082                   0.003271   
2           0.075                  0.03302                   0.000826   
3           0.100                  0.01209                   0.000302   
4           0.150                  0.00267                   0.000067   
5           0.200                  0.00084                   0.000021   

   Conditional Probability %  Unconditional Probability %  
0                     61.928                     1.548200  
1                     13.082                     0.327050  
2                      3.302                     0.082550  
3                      1.209                     0.030225  
4                      0.267                     0.006675  
5                      0.084                     0.002100  


In [ ]:

# EXPECTED EVENT FREQUENCY


probability_results["Estimated Trading Days per Event"] = (
    1 / probability_results["Unconditional Probability"]
)

print(
    probability_results[
        [
            "Loss Threshold",
            "Unconditional Probability %",
            "Estimated Trading Days per Event"
        ]
    ]
)

   Loss Threshold  Unconditional Probability %  \
0           0.030                     1.548200   
1           0.050                     0.327050   
2           0.075                     0.082550   
3           0.100                     0.030225   
4           0.150                     0.006675   
5           0.200                     0.002100   

   Estimated Trading Days per Event  
0                         64.591138  
1                        305.763645  
2                       1211.387038  
3                       3308.519438  
4                      14981.273408  
5                      47619.047619  


In [ ]:

# EXPECTED SHORTFALL


es_confidence_levels = [0.99, 0.995, 0.999]

es_results = []

for confidence in es_confidence_levels:

    var = np.quantile(
        simulated_losses,
        confidence
    )

    tail_losses = simulated_losses[
        simulated_losses >= var
    ]

    expected_shortfall = tail_losses.mean()

    es_results.append({
        "Confidence": confidence,
        "VaR": var,
        "Expected Shortfall": expected_shortfall
    })

es_results = pd.DataFrame(es_results)

es_results["VaR %"] = es_results["VaR"] * 100
es_results["Expected Shortfall %"] = (
    es_results["Expected Shortfall"] * 100
)

print(es_results)

   Confidence       VaR  Expected Shortfall      VaR %  Expected Shortfall %
0       0.990  0.105022            0.142467  10.502245             14.246735
1       0.995  0.126035            0.171105  12.603537             17.110510
2       0.999  0.188260            0.257051  18.825980             25.705057


In [ ]:

# RISK MANAGEMENT INPUTS


capital = 100_000

risk_limit_pct = 0.02

confidence_level = 0.99

In [ ]:
risk_budget = capital * risk_limit_pct

print(f"Portfolio Capital: ${capital:,.2f}")
print(f"Risk Limit: {risk_limit_pct:.2%}")
print(f"Maximum Risk Budget: ${risk_budget:,.2f}")

Portfolio Capital: $100,000.00
Risk Limit: 2.00%
Maximum Risk Budget: $2,000.00


In [ ]:
selected_var = np.quantile(
    simulated_losses,
    confidence_level
)

print(f"Selected Confidence Level: {confidence_level:.2%}")
print(f"EVT VaR: {selected_var:.4%}")

Selected Confidence Level: 99.00%
EVT VaR: 10.5022%


In [ ]:
position_size = risk_budget / selected_var

print(f"Maximum Position Size: ${position_size:,.2f}")

Maximum Position Size: $19,043.55


In [1]:

# FINAL RISK REPORT


var_dollar = position_size * selected_var

selected_es = es_results.loc[
    es_results["Confidence"] == confidence_level,
    "Expected Shortfall"
].iloc[0]

es_dollar = position_size * selected_es

print("\n" + "=" * 60)
print("EVT RISK MANAGEMENT REPORT")
print("=" * 60)

print(f"Portfolio Capital:       ${capital:,.2f}")
print(f"Risk Limit:              {risk_limit_pct:.2%}")
print(f"Risk Budget:             ${risk_budget:,.2f}")
print(f"Confidence Level:        {confidence_level:.2%}")
print(f"EVT VaR:                 {selected_var:.4%}")
print(f"Maximum Position Size:   ${position_size:,.2f}")
print(f"VaR Dollar Loss:         ${var_dollar:,.2f}")
print(f"Expected Shortfall:      {selected_es:.4%}")
print(f"ES Dollar Loss:          ${es_dollar:,.2f}")
print("=" * 60)

NameError: name 'position_size' is not defined